In [24]:
from mistralai.client import Mistral
import os
import pandas as pd 
import faiss
import numpy as np
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time


In [25]:
# Initialiser l'encodeur (utilisez le modèle approprié)
encoding = tiktoken.get_encoding("cl100k_base")  # Encodeur standard pour la plupart des modèles

# Calculer les tokens pour une description
def count_tokens(text):
    return len(encoding.encode(text))

In [26]:
# Load file
source_data = pd.read_json('../data/evenements-publics-openagenda_26.json')

print(f'Nombre de lignes avant filtrage {source_data.shape[0]}')

# Filtrer les lignes de source_data avec des descriptions vide ou avec uniquement des espaces
source_data = source_data[source_data["longdescription_fr"].apply(lambda x: isinstance(x, str) and bool(x.strip()))]

print(f'Nombre de lignes après filtrage {source_data.shape[0]}')

long_desc_list = source_data["longdescription_fr"].tolist()

print(f'Nombre de lignes dans long_desc_list {len(long_desc_list)}')


Nombre de lignes avant filtrage 6925
Nombre de lignes après filtrage 6628
Nombre de lignes dans long_desc_list 6628


In [28]:
# Filtrer uniquement des événements avec des dates de début à mai 2026.
# Convert both dates to UTC timezone for comparison
source_data = source_data[
    pd.to_datetime(source_data["firstdate_begin"], utc=True) <= pd.to_datetime("2026-05-01", utc=True)
]

print(f'Nombre de lignes après filtrage des événements avec des dates de début à mai 2026 {source_data.shape[0]}')

Nombre de lignes après filtrage des événements avec des dates de début à mai 2026 4248


In [29]:
# Analyser vos descriptions
source_data["token_count"] = source_data["longdescription_fr"].apply(count_tokens)

# Statistiques
print(f"Nombre moyen de tokens: {source_data['token_count'].mean():.2f}")
print(f"Nombre max de tokens: {source_data['token_count'].max()}")
print(f"Nombre min de tokens: {source_data['token_count'].min()}")

# Afficher la distribution
print("\nDistribution des tokens:")
print(source_data["token_count"].describe())

# Voir les descriptions les plus longues
print("\nTop 5 descriptions les plus longues:")
print(source_data.nlargest(5, "token_count")[["uid", "longdescription_fr", "token_count"]])

Nombre moyen de tokens: 151.85
Nombre max de tokens: 1983
Nombre min de tokens: 8

Distribution des tokens:
count    4248.000000
mean      151.853343
std       136.893177
min         8.000000
25%        87.000000
50%       131.000000
75%       153.000000
max      1983.000000
Name: token_count, dtype: float64

Top 5 descriptions les plus longues:
           uid                                 longdescription_fr  token_count
6735  47265444  <p><strong>Programme de la Fête de la musique ...         1983
1973  32423719  <p><strong>Au Magasin</strong><br><em>- Christ...         1863
4531  90166892  <p>Nous avons la grande chance de recevoir <a ...         1837
6732  81819977  <p>Ne manquez pas l'Evènement BROC' LAND GEEK ...         1640
3685   3298604  <h2><strong>Salon de l'Erotisme au Parc des Ex...         1630


## Chunking

Après analyse des données, le constat est le suivant : 
- 75% des descriptions < 153 tokens → Pas besoin de chunking
- Max 1983 tokens → Quelques descriptions très longues nécessitent un découpage
- Moyenne 151 tokens → Largement sous la limite de 512 tokens

Conclusion, nous allons utiliser le chunking conditionnel en découpant uniquement les descriptions qui sont supérieures à 512 tokens.

In [30]:
# Configuration du splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # Taille cible en tokens (marge de sécurité)
    chunk_overlap=50,  # Chevauchement pour garder le contexte
    length_function=count_tokens,
    separators=["\n\n", "\n", ". ", " ", ""]  # Découpe intelligente
)

def process_description(row):
    """Découpe uniquement si nécessaire"""
    text = row["longdescription_fr"]
    token_count = row["token_count"]
    uid = row["uid"]
    firstdate_begin = row["firstdate_begin"]
    firstdate_end = row["firstdate_end"]
    lastdate_begin = row["lastdate_begin"]
    lastdate_end = row["lastdate_end"]
    
    if token_count <= 512:
        # Pas de chunking nécessaire
        return [{
            "uid": uid,
            "chunk_id": 0,
            "text": text,
            "token_count": token_count,
            "firstdate_begin": firstdate_begin,
            "firstdate_end": firstdate_end,
            "lastdate_begin": lastdate_begin,
            "lastdate_end": lastdate_end
        }]
    else:
        # Chunking pour les longues descriptions
        chunks = text_splitter.split_text(text)
        return [{
            "uid": uid,
            "chunk_id": i,
            "text": chunk,
            "token_count": count_tokens(chunk),
            "firstdate_begin": firstdate_begin,
            "firstdate_end": firstdate_end,
            "lastdate_begin": lastdate_begin,
            "lastdate_end": lastdate_end
        } for i, chunk in enumerate(chunks)]

# Appliquer le traitement
chunks_list = []
for _, row in source_data.iterrows():
    chunks_list.extend(process_description(row))

# Créer un nouveau DataFrame avec les chunks
chunks_df = pd.DataFrame(chunks_list)

print(f"Nombre de descriptions originales: {len(source_data)}")
print(f"Nombre total de chunks: {len(chunks_df)}")
print(f"Descriptions découpées: {len(chunks_df[chunks_df['chunk_id'] > 0]['uid'].unique())}")

Nombre de descriptions originales: 4248
Nombre total de chunks: 4454
Descriptions découpées: 106


## Embedding
Utilisation de Mistral pour l'embedding

In [31]:
def create_smart_batches(texts, max_tokens_per_batch=8000):
    """Crée des batches en fonction du nombre de tokens"""
    batches = []
    current_batch = []
    current_tokens = 0
    
    for text in texts:
        text_tokens = count_tokens(text)
        
        if current_tokens + text_tokens > max_tokens_per_batch and current_batch:
            batches.append(current_batch)
            current_batch = [text]
            current_tokens = text_tokens
        else:
            current_batch.append(text)
            current_tokens += text_tokens
    
    if current_batch:
        batches.append(current_batch)
    
    return batches

In [32]:
texts_to_embed = chunks_df["text"].tolist()

batches = create_smart_batches(texts_to_embed, max_tokens_per_batch=7000)
print(f"Nombre de batches créés: {len(batches)}")

all_embeddings = []

with Mistral(
    api_key=os.getenv("MISTRAL_API_KEY", ""),
) as mistral:
    
    for i, batch in enumerate(batches):
        print(f"Batch {i+1}/{len(batches)} - {len(batch)} textes")
        
        try:
            res = mistral.embeddings.create(
                model="mistral-embed", 
                inputs=batch
            )
            
            all_embeddings.extend([item.embedding for item in res.data])
            print(f"  ✓ Total embeddings accumulés: {len(all_embeddings)}")
            
            # Pause entre les batches pour respecter le rate limit
            if i < len(batches) - 1:  # Pas de pause après le dernier batch
                wait_time = 2  # Secondes entre chaque batch
                print(f"  ⏳ Pause de {wait_time}s...")
                time.sleep(wait_time)
                
        except Exception as e:
            if "429" in str(e) or "rate_limited" in str(e).lower():
                print(f"  ⚠️ Rate limit atteint, pause de 60s...")
                time.sleep(60)
                # Réessayer le batch
                print(f"  🔄 Nouvelle tentative pour le batch {i+1}")
                res = mistral.embeddings.create(
                    model="mistral-embed", 
                    inputs=batch
                )
                all_embeddings.extend([item.embedding for item in res.data])
                print(f"  ✓ Batch traité après retry")
            else:
                print(f"  ✗ Erreur: {e}")
                raise

# Vérification avant assignation
print(f"\nVérification:")
print(f"  - Nombre de chunks: {len(chunks_df)}")
print(f"  - Nombre d'embeddings: {len(all_embeddings)}")

# Ajouter les embeddings au DataFrame de chunks
chunks_df["embedding"] = all_embeddings

# Sauvegarder les métadonnées pour la recherche
chunks_df.to_json("chunks_with_embeddings.json", orient="records", force_ascii=False)

Nombre de batches créés: 94
Batch 1/94 - 41 textes
  ✓ Total embeddings accumulés: 41
  ⏳ Pause de 2s...
Batch 2/94 - 46 textes
  ✓ Total embeddings accumulés: 87
  ⏳ Pause de 2s...
Batch 3/94 - 55 textes
  ✓ Total embeddings accumulés: 142
  ⏳ Pause de 2s...
Batch 4/94 - 45 textes
  ✓ Total embeddings accumulés: 187
  ⏳ Pause de 2s...
Batch 5/94 - 43 textes
  ✓ Total embeddings accumulés: 230
  ⏳ Pause de 2s...
Batch 6/94 - 48 textes
  ✓ Total embeddings accumulés: 278
  ⏳ Pause de 2s...
Batch 7/94 - 48 textes
  ✓ Total embeddings accumulés: 326
  ⏳ Pause de 2s...
Batch 8/94 - 45 textes
  ✓ Total embeddings accumulés: 371
  ⏳ Pause de 2s...
Batch 9/94 - 38 textes
  ✓ Total embeddings accumulés: 409
  ⏳ Pause de 2s...
Batch 10/94 - 56 textes
  ✓ Total embeddings accumulés: 465
  ⏳ Pause de 2s...
Batch 11/94 - 51 textes
  ✓ Total embeddings accumulés: 516
  ⏳ Pause de 2s...
Batch 12/94 - 41 textes
  ✓ Total embeddings accumulés: 557
  ⏳ Pause de 2s...
Batch 13/94 - 42 textes
  ✓ Total e

In [33]:
# Initialisation de l'index Faiss
embeddings_array = np.array([item for item in chunks_df["embedding"]])
dimension = embeddings_array.shape[1]  # Nombre de features
n_vectors = embeddings_array.shape[0]
print(f"Dimension des vecteurs: {dimension}")
print(f"Nombre de vecteurs à indexer: {n_vectors}")

# ✅ OPTIMISATION: Choix de l'index selon la taille
if n_vectors < 10000:
    # Pour petits datasets: IndexFlatL2 (exact, rapide)
    index = faiss.IndexFlatL2(dimension)
    print("Index utilisé: IndexFlatL2 (recherche exacte)")
else:
    # Pour grands datasets: IndexIVFFlat (approximatif, plus rapide)
    nlist = min(100, n_vectors // 39)  # Nombre de clusters
    quantizer = faiss.IndexFlatL2(dimension)
    index = faiss.IndexIVFFlat(quantizer, dimension, nlist)
    
    # Entraînement de l'index
    print(f"Entraînement de l'index avec {nlist} clusters...")
    index.train(embeddings_array)
    print("Index utilisé: IndexIVFFlat (recherche approximative rapide)")

# Ajout des embeddings à l'index
index.add(embeddings_array)

# ✅ VÉRIFICATION 2: Tous les vecteurs sont dans l'index
assert index.ntotal == n_vectors, f"Index incomplet! {index.ntotal}/{n_vectors}"
print(f"✓ Vérification: {index.ntotal} vecteurs indexés sur {n_vectors}")

# Sauvegarde de l'index sur le disque
faiss.write_index(index, "faiss_index.idx")

# ✅ VÉRIFICATION 3: Test de cohérence
print("\n=== Test de cohérence ===")
test_vector = embeddings_array[0:1]
distances, indices = index.search(test_vector, 1)
assert indices[0][0] == 0, "L'index ne retourne pas le bon vecteur!"
print("✓ Test de cohérence réussi")

# Statistiques finales
print(f"\n=== Statistiques ===")
print(f"Événements originaux: {len(source_data)}")
print(f"Chunks créés: {len(chunks_df)}")
print(f"Vecteurs indexés: {index.ntotal}")
print(f"Taux de couverture: {(index.ntotal / len(source_data)) * 100:.1f}%")

Dimension des vecteurs: 1024
Nombre de vecteurs à indexer: 4454
Index utilisé: IndexFlatL2 (recherche exacte)
✓ Vérification: 4454 vecteurs indexés sur 4454

=== Test de cohérence ===
✓ Test de cohérence réussi

=== Statistiques ===
Événements originaux: 4248
Chunks créés: 4454
Vecteurs indexés: 4454
Taux de couverture: 104.8%


In [34]:
# ✅ VÉRIFICATION 4: Distribution des distances
print("\n=== Analyse de la qualité de l'index ===")

# Recherche de tous les vecteurs dans l'index
all_distances, all_indices = index.search(embeddings_array[:100], 1)

print(f"Distance moyenne à soi-même: {all_distances.mean():.6f}")
print(f"Distance max à soi-même: {all_distances.max():.6f}")

if all_distances.mean() > 0.001:
    print("⚠️ Attention: Les vecteurs ne se retrouvent pas exactement")
else:
    print("✓ Qualité de l'index: Excellente")

# ✅ VÉRIFICATION 5: Couverture des événements originaux
indexed_uids = set(chunks_df['uid'].unique())
original_uids = set(source_data['uid'].unique())

missing_uids = original_uids - indexed_uids
if missing_uids:
    print(f"⚠️ {len(missing_uids)} événements manquants: {list(missing_uids)[:5]}")
else:
    print(f"✓ Tous les {len(original_uids)} événements sont indexés")



=== Analyse de la qualité de l'index ===
Distance moyenne à soi-même: 0.000000
Distance max à soi-même: 0.000000
✓ Qualité de l'index: Excellente
✓ Tous les 4248 événements sont indexés


In [23]:
# Exemple de recherche
def search_events(query, k=5):
    """Recherche les k événements les plus similaires"""
    
    # 1. Vectoriser la requête
    with Mistral(api_key=os.getenv("MISTRAL_API_KEY", "")) as mistral:
        query_res = mistral.embeddings.create(
            model="mistral-embed",
            inputs=[query]
        )
        query_embedding = np.array([query_res.data[0].embedding])
    
    # 2. Rechercher dans FAISS
    distances, indices = index.search(query_embedding, k)
    
    # 3. Récupérer les résultats
    results = chunks_df.iloc[indices[0]]
    
    return results[["uid", "chunk_id", "text", "token_count"]], distances[0]

# Test
results, distances = search_events("concert de musique classique", k=5)
print("Top 5 événements similaires:")
print(results)
print(f"\nDistances: {distances}")

Top 5 événements similaires:
           uid  chunk_id                                               text  \
5239  32732635         0  <p>Concert acoustique de musique trad irlandai...   
2190  90148048         0  <p><strong>Concert à Aix-en-Provence : Shostak...   
5092  55339912         0  <h3><strong>Un concert éclatant à Nice 🌸💃</str...   
45    85623782         0  <p>Venez tester vos connaissances en musique d...   
6507  74877992         0  <p>Ouverture de l’établissement de 10h à 12h e...   

      token_count  
5239           16  
2190          349  
5092          373  
45             25  
6507          286  

Distances: [0.4542258  0.45462716 0.4599963  0.47203267 0.47652996]
